In [1]:
import os
import glob

from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_30460\2162706441.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
D:\Works\govfund_agent\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ---------------------------------------------------------------------------
# 설정값
# ---------------------------------------------------------------------------
DATA_DIR = "../rag_data"
DB_DIR = "../chroma_govfund_db"
COLLECTION_NAME = "govfund_guide"
EMBEDDING_MODEL = "bge-m3"  # 사용자 환경에 맞게 변경 (ollama pull bge-m3 필요)
CHUNK_SIZE = 400
CHUNK_OVERLAP = 50

# 파일명 -> 카테고리 매핑 (post_processing / routing 에서 함께 사용)
CATEGORY_MAP = {
    "(공고문)_2026년도_중앙부처_및_지자체_창업지원사업_통합공고문(제2025-648호,_2025.12.19.).pdf": "notice",
}


In [3]:
# ---------------------------------------------------------------------------
# 1. pdf 문서 로딩 -> Document 생성
# ---------------------------------------------------------------------------

documents: list[Document] = []
pdf_files = sorted(glob.glob(os.path.join(DATA_DIR, "*.pdf")))

if not pdf_files:
    raise FileNotFoundError(
        f"'{DATA_DIR}' 폴더에서 pdf 파일을 찾을 수 없습니다. "
        "rag_data 폴더 위치를 확인하세요."
    )

for path in pdf_files:
    filename = os.path.basename(path)

    loader_notice = PyPDFLoader(path)

    pdf_docs = loader_notice.load()

    category = CATEGORY_MAP.get(filename, "general")

    for doc in pdf_docs:
        doc.metadata["category"] = category

    documents.extend(pdf_docs)

    total_chars = sum(len(doc.page_content) for doc in pdf_docs)

    print(f"  - 로드 완료: {filename} ({total_chars:,}자, category={category})")
    print(f"마지막 컨텐츠: {doc.page_content}")


  - 로드 완료: (공고문)_2026년도_중앙부처_및_지자체_창업지원사업_통합공고문(제2025-648호,_2025.12.19.).pdf (105,493자, category=notice)
마지막 컨텐츠: - 105 -
연번 지역
구분 사업명 사업개요 지원내용 지원대상 예산
(억원)
사업
공고일 소관 부처 전담(주관)
기관 비고
16 충남 ㆍ천안 글 로 벌  액 셀 러
레 이 팅 지 원 사 업
해 외 진 출  역 량  강 화  교 육, 
글 로 벌  전 시 회  참 가 지 원
①해 외  전 시 회 
참 가 지 원,
②투 자·자 금  연 계, 
③글 로 벌  진 출  지 원
천 안  관 내  사 업 장 을 
보유한 7년이내 
스타트업
4.52 ’26.2월 천안시
(미 래 전 략 과)
천안
과학산업
진흥원
해외
진출
17 경북
ㆍ경북 글로벌 
스타트업 패키지 
성장지원
세 계 시 장 을  선 도 할  첨 단 
기 술 아 이 템 을  보 유 한 
벤 처·스 타 트 업  육 성 을  통 한 
혁 신  기 술 창 업  성 공 률  제 고 
및  글 로 벌  시 장  진 출  지 원
①사 업 화  지 원
②멘 토 링‧컨 설 팅·교 육
③글 로 벌해 외 현 지 
프 로 그 램  지 원
경북 소재 7년 
이내 창업기업  
(단, 신산업분야 창업 
10년이내)
3.5 ’26.3월 경상북도
(기업지원과)
(재)경북창조
경제혁신센터
(창업성장팀)
-
18 경북
ㆍ구미시 스타트업 
해외시장 개척 
지원사업
스 타 트 업 의  해 외  전 시 회  등 
글 로 벌  시 장 진 출 을  위 한  지 원해 외 시 장  진 출  지 원7년 미만 
창업기업 1.2 ’26.1월 구미시
(기업지원과)
구 미
전 자 정 보 기 술 원
(창 업 성 장 지 원 센 터)
-
19 전북 ㆍ해 외  스 타 트 업  유 치 
지 원 사 업
「K-Startup 그 랜 드  챌 린 지*」 
선 정  우 수  해 외  스 타 트 업 
도 내  유 치  및  정 착  지 원
 * 중 소 벤 처 기 업 부  주 최


In [4]:
keyword = "중소벤처"

for i, doc in enumerate(pdf_docs, start=1):
    if keyword in doc.page_content:
        print(f"{i}페이지에서 발견")
        print(doc.page_content)

1페이지에서 발견
- 1 -
[중소벤처기업부 공고 제2025-648호]
2026년 중앙부처 및 지자체 창업지원사업 통합공고
 ｢중소기업창업지원법｣ 제14조(창업정책정보의수집 및 제공)에 따라
2026년 창업지원사업을 공고하오니, 창업기업 및 예비창업자의 적극
적인참여를 바랍니다.
2025년 12월 19일
중소벤처기업부장관
1  개 요
 ◦ 근거 : ｢
중소기업창업지원법｣
제14조
 ◦ 목적 :창업자및 예비창업자가국내창업지원사업 정보를알기
쉽게접할수있도록중앙부처및지자체창업지원사업통합공고
 ◦경과
 - (‘16)중앙부처창업지원사업통합공고실시(6개기관, 65개사업, 0.6조원)
 - (‘21)광역지자체창업지원사업추가(31개기관, 193개사업, 1.5조원)
 - (‘22)기초지자체창업지원사업및융자사업추가
 - (‘23) 창업지원사업 8개유형별로구분
    * 사업화, 시설·공간·보육, 멘토링·컨설팅·교육, 행사·네트워크, 글로벌 진출, 융자, 기술
개발(R&D), 인력
 - (‘26)창업지원사업유형중융자·보증유형에중앙부처보증사업추가
   < ’16~‘26년 창업지원사업 통합공고 현황> 
구분 참여기관 사업규모
‘16년 6개 중앙부처 65개 사업, 5,764억원
‘17년 7개 중앙부처 62개 사업, 6,158억원
‘18년 7개 중앙부처 60개 사업, 7,796억원
‘19년 14개 중앙부처 69개 사업, 11,181억원
‘20년 16개 중앙부처 90개 사업, 14,517억원
‘21년 14개 중앙부처
17개 광역지자체
89개 사업, 13,812억원(중앙부처)
104개 사업, 811억원(광역지자체)
5페이지에서 발견
- 5 -
3  사업별 주요 내용
 중앙부처
연
번 사업명 사업개요 지원내용 지원대상 예산
(억원)
사업
공고일 소관 부처 전문(주관)
기관 비고
◇ 사업화 (33건)
1 ㆍ예비창업패키지
혁 신 적 인  기 술 창 업  아 이 디 어 를 
보유한 예비창업자의 창업
사업화 준비단계를 지원하여 
성 공 적 인  창 업 시 장  안 착  유 도
①사업화 자금
②창업

In [5]:
# ---------------------------------------------------------------------------
# 2. Chunk 분할
# ---------------------------------------------------------------------------

print("all_docs:",len(documents))

splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE,chunk_overlap=CHUNK_OVERLAP)

chunks = splitter.split_documents(documents)

# chunk 순번 metadata 부여 (디버깅/추적용)
for i, chunk in enumerate(chunks):
    chunk.metadata["chunk_index"] = i

print(len(chunks))


all_docs: 105
336


In [6]:
# ---------------------------------------------------------------------------
# 3~4. Embedding 생성 + Chroma Vector DB 저장
# ---------------------------------------------------------------------------
import shutil
embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL,base_url="http://10.8.0.1:11434")

if os.path.exists(DB_DIR):
    shutil.rmtree(DB_DIR)
    print(f"\n기존 '{DB_DIR}' 폴더가 존재합니다. 동일 컬렉션을 다시 생성/덮어씁니다.")


vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name=COLLECTION_NAME,
    persist_directory=DB_DIR,
)

print("all_docs :", len(documents))
print("chunks   :", len(chunks))
print("chroma   :", vectorstore._collection.count())



all_docs : 105
chunks   : 336
chroma   : 336


In [7]:
base_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4},
)

docs = base_retriever.invoke("중소벤처")

print("검색 결과:", len(docs))

for i, doc in enumerate(docs, 1):
    print(f"\n--- {i} ---")
    print(doc.page_content[:500])
    print(doc.metadata)

검색 결과: 4

--- 1 ---
벤처기업 대상 기술개발
R&D 지원 및 사업화 지원
①기술개발R&D
②사업화자금
연 매출액 120억원
이하의 관내 제조
벤처기업
47.5
‘26.2~
3월
(예정)
인천광역시
(창업벤처과)
인천
테크노파크
-
3 경기
ㆍ경기도 시스템
반도체 OSAT 
분야 기술개발
지원사업
한국나노기술원에 기 구축한
R&D Foundry 지원과 더불어
시스템반도체 통합솔루션 
기반의 OSAT를 동시지원
하여  시스템반도체 기업의
①사업화 자금
②기술컨설팅
③나 노 팹  입 주 지 원, 
OSAT 전 문  교 육 
지 원, 기 술 교 류 회 
예비창업자 및 
창 업(10년  이 내) 기 업 3.9
‘26.2~
3월
(예정)
경기도
(반도체
산업과)
한국
나노기술원
(대외협력처)
-
{'chunk_index': 166, 'total_pages': 105, 'page': 50, 'moddate': '2025-12-22T13:28:06+09:00', 'author': 'mss', 'page_label': '51', 'pdfversion': '1.4', 'creationdate': '2025-12-22T13:28:06+09:00', 'source': '../rag_data\\(공고문)_2026년도_중앙부처_및_지자체_창업지원사업_통합공고문(제2025-648호,_2025.12.19.).pdf', 'producer': 'Hancom PDF 1.3.0.550', 'category': 'notice', 'creator': 'Hwp 2018 10.0.0.14515'}

--- 2 ---
한 규정의 27개 분야 
스타트업
11 ‘26.1월
중소벤처
기업부
(창업정책과)
창업진흥원
(원스톱지원실)
신산업, 
규제
해소
3 ㆍ스타트업 AI 기
술 인력 양성
혁신 벤처·스타트업이 
필요로 하는 인공지능 
실무교육 제공 및 취·창업 
연계지원
①인공지능 
특화교육
②취·창업 연계
지원
만 39세 이하의 청년
(학력·전공 무관) 29 ’25.12월 중기부
(청 년 정 